# 00 · 先看資料：iris 探索（EDA）

> **這本為什麼是 notebook**：資料科學家動手的第一件事是「看」——分布長怎樣、類別平不平衡、哪些特徵分得開。
> 這些理解來自圖，不是 `print()`。
>
> **配套腳本**：同資料夾的 `01_baseline_iris.py` 刻意維持 `.py`。它的主題是**可重現**
> （固定 seed，今天跑、明天跑、別人跑都一樣），而那正是 notebook 最弱的地方。
> 兩者的介質差異本身就是一課，判準見 [`docs/notebook-vs-script.md`](../../../../docs/notebook-vs-script.md)。
>
> **順序**：先跑本 notebook 建立對資料的直覺 → 再跑 `01_baseline_iris.py` 得到可重現的 baseline。

> 圖上的文字一律用英文：教學機器不一定裝了中文字型，用英文避免變成一排方框。

In [ ]:
from pathlib import Path


def find_course_root() -> Path:
    """從當前目錄往上找到 mlops-course 根目錄（含 datasets/ 的那層）。

    notebook 沒有 __file__，而且你可能從任何位置啟動 Jupyter，
    所以用「往上層找標記檔」定位，不寫死相對路徑。
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "datasets" / "iris.csv").exists():
            return base
    raise FileNotFoundError("找不到 mlops-course/datasets/，請在 mlops-course/ 之內開啟本 notebook")


ROOT = find_course_root()
print("course root =", ROOT)

import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(ROOT / "datasets" / "iris.csv")
print(f"形狀：{df.shape[0]} 列 × {df.shape[1]} 欄")
df.head()

## 1. 這份資料長什麼樣

先看型別與統計摘要。**型別**決定能做什麼運算，**摘要**告訴你尺度差多少
（尺度差很多時，之後某些模型需要標準化）。

In [ ]:
display(df.dtypes.to_frame("dtype"))

FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
df[FEATURES].describe().T.round(2)

## 2. 類別平衡：三種花各幾筆？

**為什麼先看這個**：類別嚴重不平衡時，accuracy 會說謊
（99% 都是 A 的資料，模型全猜 A 就有 99% 準確率，但它什麼都沒學到）。
先確認平衡，才知道 accuracy 能不能當主要指標。

In [ ]:
counts = df["target_name"].value_counts().sort_index()
print(counts.to_string())

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(counts.index, counts.values, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_ylabel("count")
ax.set_title("Class balance")
for i, v in enumerate(counts.values):
    ax.text(i, v + 1, str(v), ha="center")
plt.tight_layout()
plt.show()

完全平衡（各 50 筆）。**結論：accuracy 在這份資料上是誠實的指標**，
不需要改用 F1 或加權——這個判斷就是 `01_baseline_iris.py` 直接用 accuracy 的依據。

## 3. 特徵分布：哪些特徵把類別分得開？

每個特徵畫一張「依類別分色的直方圖」。
**看的是重疊程度**：三色分得越開的特徵，對分類越有用。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
colors = {"setosa": "#4C78A8", "versicolor": "#F58518", "virginica": "#54A24B"}

for ax, col in zip(axes.ravel(), FEATURES):
    for name, sub in df.groupby("target_name"):
        ax.hist(sub[col], bins=15, alpha=0.6, label=name, color=colors[name])
    ax.set_title(col)
    ax.set_xlabel("value")
    ax.set_ylabel("count")

axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

讀圖：`petal_length` 與 `petal_width` 上，**setosa 幾乎完全分離**（藍色自成一區）；
`sepal_width` 三色重疊最嚴重，資訊量最低。

## 4. 兩兩關係：最能分類的是哪一組？

把最有希望的兩個特徵畫成散布圖，直接看類別是否可分。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for name, sub in df.groupby("target_name"):
    axes[0].scatter(sub["petal_length"], sub["petal_width"],
                    label=name, alpha=0.7, color=colors[name])
    axes[1].scatter(sub["sepal_length"], sub["sepal_width"],
                    label=name, alpha=0.7, color=colors[name])

axes[0].set_xlabel("petal_length"); axes[0].set_ylabel("petal_width")
axes[0].set_title("Petal: nearly separable")
axes[1].set_xlabel("sepal_length"); axes[1].set_ylabel("sepal_width")
axes[1].set_title("Sepal: heavily overlapped")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

左圖幾乎能用兩條直線切開三類 → **這份資料對線性模型很友善**，
所以 baseline 選 `LogisticRegression` 是合理的，不需要一開始就上複雜模型。

## 5. 相關性：有沒有幾乎重複的特徵？

高度相關的特徵帶的是重複資訊。這在 baseline 階段影響不大，
但到了特徵工程（m3）會決定你要保留哪些、丟掉哪些。

In [ ]:
corr = df[FEATURES].corr()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURES)), FEATURES, rotation=45, ha="right")
ax.set_yticks(range(len(FEATURES)), FEATURES)
for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature correlation")
plt.tight_layout()
plt.show()

`petal_length` 與 `petal_width` 相關 0.96——**幾乎是同一個訊號**。
真實專案裡這是「可以只留一個」的候選。

## 6. EDA 結論 → 交棒給 `01_baseline_iris.py`

從上面四張圖，我們得到三個**會影響下一步決策**的結論：

| 觀察 | 對下一步的意義 |
| :--- | :--- |
| 三類各 50 筆，完全平衡 | accuracy 可以當主要指標 |
| 花瓣特徵近乎線性可分 | baseline 用 LogisticRegression 就夠，不必一開始上複雜模型 |
| 花瓣長寬相關 0.96 | 特徵有冗餘，m3 做特徵工程時再處理 |

**接下來換介質**：探索到此為止，下一步是產出一個「今天跑、明天跑、別人跑都給同一個數字」的 baseline。
那件事需要固定 seed、需要被 CI 跑、需要能被 diff——所以它是一支腳本，不是這本 notebook：

```bash
python 01_baseline_iris.py
```

> 這就是本課程反覆出現的節奏：**在 notebook 想清楚，在腳本固定下來。**